# JSONL LLM Trace Explorer

Interactive viewer for task-decomposition JSONL event logs. Run all cells, enter the JSONL file path when prompted, and the trace displays below. Supports any similarly structured JSONL file. No network access required.

**Install (only if needed):** `uv add --dev jupyterlab ipykernel notebook pandas`  
For VS Code, open this notebook with a Python/Jupyter kernel.

In [7]:
from pathlib import Path
import json, html
import pandas as pd
from IPython.display import display, HTML, JSON, Markdown

TRACE_PATH = Path(input('Enter the JSONL trace file path: ').strip()).expanduser()
if not TRACE_PATH.is_file():
    raise FileNotFoundError(f'Trace file not found: {TRACE_PATH}')


In [8]:
def load_trace(path):
    events, issues = [], []
    with Path(path).open(encoding='utf-8-sig') as f:
        for line_number, line in enumerate(f, 1):
            if not line.strip(): continue
            try:
                item = json.loads(line)
                if not isinstance(item, dict):
                    issues.append(f'Line {line_number}: expected a JSON object')
                    continue
                item['_line'] = line_number
                events.append(item)
            except json.JSONDecodeError as e:
                issues.append(f'Line {line_number}: {e}')
    return events, issues

def stage_of(e): return e.get('stage') or '(session)'
def status_of(e):
    if e.get('result') is not None: return str(e['result'])
    if e.get('event') == 'execution.summary': return str(e.get('data',{}).get('status','—'))
    return '—'
def as_table(events):
    return pd.DataFrame([{
        'line':e['_line'], 'timestamp':e.get('timestamp',''),
        'stage':stage_of(e), 'event':e.get('event',''),
        'status':status_of(e),
        'latency_ms':e.get('data',{}).get('latency_ms'),
        'model':e.get('data',{}).get('model') or e.get('data',{}).get('response_metadata',{}).get('model','') if isinstance(e.get('data',{}).get('response_metadata',{}),dict) else '',
    } for e in events])

events, issues = load_trace(TRACE_PATH)
df = as_table(events)
print(f'Loaded {len(events)} events from {TRACE_PATH.name}; parse issues: {len(issues)}')
if issues: display(pd.DataFrame({'issue':issues}))


Loaded 21 events from task_decomposition_2026-10-15_001927.jsonl; parse issues: 0


## Execution summary

In [9]:
display(pd.DataFrame([{'events':len(events), 'stages':df.loc[df.stage!='(session)','stage'].nunique(),
                       'failures':int(df.status.isin(['failure','error','failed']).sum()),
                       'total_llm_latency_s':round(pd.to_numeric(df.latency_ms,errors='coerce').sum()/1000,2)}]))
display(df.groupby('stage',sort=False).agg(events=('event','size'), last_status=('status','last'), llm_latency_ms=('latency_ms','sum')).reset_index())

def usage_rows(events):
    rows=[]
    for e in events:
        usage=e.get('data',{}).get('usage')
        if e.get('event')!='llm.response' or not isinstance(usage,dict):continue
        rows.append({'stage':stage_of(e),**{k:v for k,v in usage.items() if isinstance(v,(int,float))}})
    return pd.DataFrame(rows)
usage=usage_rows(events)
if not usage.empty:
    display(Markdown('### Token usage by response'))
    display(usage)


,events,stages,failures,total_llm_latency_s
0,21,4,0,19.49


,stage,events,last_status,llm_latency_ms
0,(session),1,—,0.000
1,operational_decomposition,5,success,10281.690
2,semantic_evaluation,4,accept,1316.387
3,retain_remove_classification,5,success,6181.335
4,added_work_classification,6,success,1711.234


### Token usage by response

,stage,input_tokens,output_tokens,total_tokens
0,operational_decomposition,786,746,1532
1,semantic_evaluation,2135,37,2172
2,retain_remove_classification,1461,582,2043
3,added_work_classification,2267,72,2339


## Human-readable trace
Each JSONL event is shown below as a collapsible, formatted section.

In [10]:
def pretty(value):
    if isinstance(value,str):
        try: value=json.loads(value)
        except (ValueError,TypeError): return value
    return json.dumps(value,ensure_ascii=False,indent=2,default=str)

def panel(title,value,open_by_default=False):
    content=html.escape(pretty(value))
    return f'<details {"open" if open_by_default else ""}><summary style="cursor:pointer;font-weight:600;padding:7px">{html.escape(title)}</summary><pre style="white-space:pre-wrap;overflow-wrap:anywhere;background:#f5f7fa;color:#182230;padding:12px;border-radius:6px;max-height:520px;overflow:auto">{content}</pre></details>'

def show_event(e):
    data=e.get('data') or {}
    title=f"Line {e['_line']} · {e.get('event','')} · {stage_of(e)} · {status_of(e)}"
    display(Markdown('### '+title))
    sections=[]
    if isinstance(data,dict):
        if 'messages' in data:
            for idx,msg in enumerate(data['messages']):
                sections.append(panel(f"Message {idx+1} — {msg.get('role','')}",msg.get('content','')))
        for key in ('parsed_response','response','raw_output','errors','error','usage','parameters','structured_output_schema','raw_response'):
            if key in data: sections.append(panel(key,data[key],key in ('errors','error','parsed_response')))
        remaining={k:v for k,v in data.items() if k not in ('messages','parsed_response','response','raw_output','errors','error','usage','parameters','structured_output_schema','raw_response')}
        if remaining: sections.append(panel('Other data',remaining))
    sections.append(panel('Complete event JSON',e))
    display(HTML(''.join(sections)))

for event in events:
    show_event(event)


### Line 1 · trace.session · (session) · —

### Line 2 · llm.request · operational_decomposition · —

### Line 3 · llm.response · operational_decomposition · success

### Line 4 · llm.parsed · operational_decomposition · —

### Line 5 · validation.structural · operational_decomposition · success

...

### Line 17 · llm.response · added_work_classification · success

### Line 18 · llm.parsed · added_work_classification · —

### Line 19 · validation.structural · added_work_classification · success

### Line 20 · validation.structural · added_work_classification · success

### Line 21 · execution.summary · added_work_classification · success

## Structured output: operational subtasks
Extracts the decomposition from parsed events, validation results, or response text, preferring parsed output.

In [11]:
def find_subtasks(obj):
    if isinstance(obj,dict):
        subtasks=obj.get('operational_subtasks')
        if isinstance(subtasks,list):return obj
        for v in obj.values():
            result=find_subtasks(v)
            if result is not None:return result
    elif isinstance(obj,list):
        for v in obj:
            result=find_subtasks(v)
            if result is not None:return result
    elif isinstance(obj,str):
        try:return find_subtasks(json.loads(obj))
        except (ValueError,TypeError):pass
    return None

candidates=sorted(events,key=lambda e: 0 if e.get('event')=='llm.parsed' else 1)
decomposition=next((found for e in candidates if (found:=find_subtasks(e.get('data',{}))) is not None),None)
if decomposition:
    task=decomposition.get('task',{})
    display(Markdown('**Task:** '+str(task.get('task_name','Unknown'))))
    subtasks=pd.DataFrame(decomposition['operational_subtasks'])
    if 'depends_on' in subtasks: subtasks['depends_on']=subtasks['depends_on'].apply(lambda x:', '.join(map(str,x)) if isinstance(x,list) else str(x))
    display(subtasks.drop(columns=['provenance_refs'],errors='ignore').style.set_properties(**{'text-align':'left','white-space':'pre-wrap'}))
else: print('No operational_subtasks object found in this trace.')


**Task:** Production Scheduling

,sequence_index,subtask_id,subtask_name,description,relative_effort_weight,depends_on
0,1,subtask-1,Review production orders,Examine the list of production orders for required products and timelines.,3,
1,2,subtask-2,Check inventory availability,Assess current inventory levels to ensure sufficient materials for production.,3,
2,3,subtask-3,Evaluate machine capacity,Determine the availability and capacity of machines needed for production tasks.,4,subtask-2
3,4,subtask-4,Assess workforce availability,Review workforce schedules and availability to ensure adequate staffing for production.,3,
4,5,subtask-5,Develop production schedule,Create a feasible production schedule based on resources and needs identified.,5,"subtask-1, subtask-2, subtask-3, subtask-4"
5,6,subtask-6,Assign production orders,Allocate production orders to specific machines and shifts based on the schedule.,4,subtask-5
6,7,subtask-7,Resolve scheduling conflicts,Identify and address any conflicts or overlaps in the production schedule.,4,subtask-6
7,8,subtask-8,Communicate finalized schedule,Distribute the completed production schedule to all relevant production teams.,3,subtask-7


## RETAIN / REMOVE classification
Shows which operational subtasks remain human-performed and which are automated or delegated.

In [12]:
def find_classification(obj):
    if isinstance(obj, dict):
        rows = obj.get('classified_subtasks')
        if isinstance(rows, list): return obj
        for value in obj.values():
            found = find_classification(value)
            if found is not None: return found
    elif isinstance(obj, list):
        for value in obj:
            found = find_classification(value)
            if found is not None: return found
    elif isinstance(obj, str):
        try: return find_classification(json.loads(obj))
        except (ValueError, TypeError): pass
    return None

classification = next((found for event in events if (found := find_classification(event.get('data', {}))) is not None), None)
if classification:
    names = {row.get('subtask_id'): row.get('subtask_name') for row in decomposition.get('operational_subtasks', [])} if decomposition else {}
    rows = []
    for row in classification['classified_subtasks']:
        decision = str(row.get('classification', '')).upper()
        rows.append({
            'subtask_id': row.get('subtask_id'),
            'subtask_name': row.get('subtask_name') or names.get(row.get('subtask_id'), ''),
            'status': decision,
            'human work retained': 'Yes' if decision == 'RETAIN' else 'No',
            'automated/delegated': 'Yes' if decision == 'REMOVE' else 'No',
        })
    display(pd.DataFrame(rows).style.set_properties(**{'text-align':'left','white-space':'pre-wrap'}))
else: print('No RETAIN/REMOVE classification found in this trace.')


,subtask_id,subtask_name,status,human work retained,automated/delegated
0,subtask-1,Review production orders,RETAIN,Yes,No
1,subtask-2,Check inventory availability,RETAIN,Yes,No
2,subtask-3,Evaluate machine capacity,RETAIN,Yes,No
3,subtask-4,Assess workforce availability,RETAIN,Yes,No
4,subtask-5,Develop production schedule,RETAIN,Yes,No
5,subtask-6,Assign production orders,RETAIN,Yes,No
6,subtask-7,Resolve scheduling conflicts,RETAIN,Yes,No
7,subtask-8,Communicate finalized schedule,RETAIN,Yes,No


## Validation diagnostics

In [13]:
diagnostics=[]
for e in events:
    if e.get('event','').startswith('validation.') or e.get('event')=='execution.summary':
        d=e.get('data') or {}
        diagnostics.append({'line':e['_line'],'stage':stage_of(e),'event':e.get('event'),'result':status_of(e),
                            'errors':pretty(d.get('errors',d.get('error','')))[:2500]})
if diagnostics: display(pd.DataFrame(diagnostics).style.set_properties(**{'white-space':'pre-wrap','text-align':'left'}))
else: print('No validation events found.')


,line,stage,event,result,errors
0,5,operational_decomposition,validation.structural,success,[]
1,6,operational_decomposition,validation.structural,success,[]
2,10,semantic_evaluation,validation.evaluation,accept,[]
3,14,retain_remove_classification,validation.structural,success,[]
4,15,retain_remove_classification,validation.structural,success,[]
5,19,added_work_classification,validation.structural,success,[]
6,20,added_work_classification,validation.structural,success,[]
7,21,added_work_classification,execution.summary,success,
